                                                                 Chapter 12 Fine-Tuning Generation Models

In [ ]:
         # Instruction Tuning with QLoRA

In [ ]:
# Import AutoTokenizer to load the model's tokenizer
from transformers import AutoTokenizer

# Import load_dataset to load a dataset from Hugging Face
from datasets import load_dataset

# Load the tokenizer and chat template from TinyLlama
template_tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

# Define a function to format each conversation
def format_prompt(example):

  # Get the conversation messages from the example
    chat = example["messages"]

    # Apply TinyLlama's chat template to the messages
    prompt = template_tokenizer.apply_chat_template(
        chat,
        tokenize=False
    )

     # Return the formatted prompt as text, not tokens
    return {"text": prompt}

# Load the UltraChat 200k dataset
dataset = (
    load_dataset(
        "HuggingFaceH4/ultrachat_200k",

        # Use the test_sft portion of the dataset
        split="test_sft"
    )

     # Shuffle the dataset using a fixed seed
    .shuffle(seed=42)

    # Select only the first 3000 examples
    .select(range(3000))
)

# Apply the format_prompt function to all examples
dataset = dataset.map(format_prompt)

# Print the formatted conversation at index 2576
print(dataset["text"][2576])

In [ ]:
   # Model Quantization

In [2]:
# Install or upgrade bitsandbytes to version 0.46.1 or newer
!pip install -U bitsandbytes>=0.46.1

In [4]:
# Import PyTorch for tensor operations and model computation
import torch

# Import AutoModelForCausalLM to load a language model for text generation
# Import AutoTokenizer to convert text into tokens
# Import BitsAndBytesConfig to configure 4-bit/8-bit quantization
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Specify the TinyLlama model to load
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# Create the configuration for reducing the model's memory usage
bnb_config = BitsAndBytesConfig(

  # Load the model using 4-bit quantization
     load_in_4bit=True,

  # Use NF4 quantization, which is designed for neural network weights
    bnb_4bit_quant_type="nf4",

  # Use float16 for calculations during model execution
    bnb_4bit_compute_dtype="float16",

   # Apply an additional quantization step to save more memory
    bnb_4bit_use_double_quant=True,
)

# Load the language model
model = AutoModelForCausalLM.from_pretrained(

    model_name,

    # Automatically place the model on available devices such as the GPU
    device_map="auto",

    # Apply the 4-bit quantization configuration
    quantization_config=bnb_config,
)

# Disable the key-value cache during training to reduce memory usage
model.config.use_cache = False

# Set the tensor parallelism value to 1
model.config.pretraining_tp = 1

# Load the tokenizer associated with the model
tokenizer = AutoTokenizer.from_pretrained(
    model_name,

     # Allow custom model/tokenizer code if required by the model
    trust_remote_code=True
)

# Set <PAD> as the padding token
tokenizer.pad_token = "<PAD>"

# Add padding to the left side of the input
tokenizer.padding_side = "left"

In [ ]:
         # LoRA Configuration

In [6]:
# Import LoRA configuration and PEFT preparation functions
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# Prepare LoRA Configuration
peft_config = LoraConfig(

      # Controls how strongly the LoRA updates affect the original model
     lora_alpha=32,

       # Randomly drops 10% of LoRA values during training to reduce overfitting
     lora_dropout=0.1,

     # Sets the rank (size) of the LoRA matrices
     r=64,

      # Do not train additional bias parameters
     bias="none",

 # Specifies that the model is a causal language model
task_type="CAUSAL_LM",


# Specifies the model layers where LoRA adapters will be added
target_modules=
# Layers to target
target_modules=[
        "k_proj",       # Key projection layer
        "gate_proj",   # Gate projection layer
        "v_proj",       # Value projection layer
        "up_proj",      # Up projection layer
        "q_proj",      # Query projection layer
        "o_proj",      # Output projection layer
        "down_proj"    # Down projection layer
    ]
)

# Prepare the 4-bit quantized model for efficient training
model = prepare_model_for_kbit_training(model)

# Add the LoRA adapters to the model
model = get_peft_model(model, peft_config)

In [ ]:
      # Training Configuration

In [7]:
from transformers import TrainingArguments

# Set the folder where training results will be saved
output_dir = "./results"

# Define the training settings
training_arguments = TrainingArguments(
    # Folder to save model checkpoints and results
    output_dir=output_dir,

    # Number of examples processed at once per GPU
    per_device_train_batch_size=2,

    # Accumulate gradients for 4 steps before updating the model
    gradient_accumulation_steps=4,

    # Use a memory-efficient AdamW optimizer
    optim="paged_adamw_32bit",

    # Set the learning rate
    learning_rate=2e-4,

    # Use cosine learning rate scheduling
    lr_scheduler_type="cosine",

    # Train for 1 complete epoch
    num_train_epochs=1,

    # Log training information every 10 steps
    logging_steps=10,

    # Use 16-bit precision to reduce memory usage
    fp16=True,

    # Reduce memory usage during training
    gradient_checkpointing=True
)

In [ ]:
     # Training


In [ ]:
# Remove the current TRL version and install TRL 0.12.2
!pip uninstall -y trl
!pip install trl==0.12.2

In [ ]:
# Import SFTTrainer for supervised fine-tuning
from trl import SFTTrainer

# Create the supervised fine-tuning trainer
trainer = SFTTrainer(
    # Use the prepared TinyLlama model
    model=model,

    # Use the formatted training dataset
    train_dataset=dataset,

    # Use the "text" column as training input
    dataset_text_field="text",

    # Use the model's tokenizer to process the text
    tokenizer=tokenizer,

    # Use the training settings defined earlier
    args=training_arguments,

    # Limit each input sequence to 512 tokens
    max_seq_length=512,

    # Use the LoRA configuration for parameter-efficient training
    peft_config=peft_config,
)

# Start training the model
trainer.train()

# Save the trained QLoRA adapter weights
trainer.model.save_pretrained("TinyLlama-1.1B-qlora")

In [ ]:
          # Merge Weights

In [ ]:
# Install torchao, a PyTorch library for model optimization and quantization
!pip install -U "torchao==0.17.0"

In [ ]:
# Import AutoPeftModelForCausalLM to load the saved PEFT/QLoRA model
from peft import AutoPeftModelForCausalLM

# Load the saved QLoRA model from the specified folder
model = AutoPeftModelForCausalLM.from_pretrained(

    # Specify the folder containing the saved QLoRA model
    "TinyLlama-1.1B-qlora",

    # Reduce CPU memory usage while loading the model
    low_cpu_mem_usage=True,

    # Automatically place the model on available devices
    device_map="auto",
)

# Merge the LoRA weights with the base model and remove the adapter
merged_model = model.merge_and_unload()

In [ ]:
# Import pipeline to easily run text generation with the model
from transformers import pipeline

# Create a prompt for the instruction-tuned model
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""

# Create a text-generation pipeline using our merged model
pipe = pipeline(

   # Specify that the task is text generation
    task="text-generation",

   # Use the merged fine-tuned model
   model=merged_model,

   # Use the tokenizer for converting text to tokens
tokenizer=tokenizer)

# Generate a response and print the generated text
print(pipe(prompt)[0]["generated_text"])

             Preference Tuning with DPO

In [ ]:
       # Templating Alignment Data

In [ ]:
# Import load_dataset to load the DPO dataset from Hugging Face
from datasets import load_dataset

# Define a function to format each example for DPO training
def format_prompt(example):
    """Format the prompt using the <|user|> template TinyLlama is using"""

     # Format the prompt using TinyLlama's chat template
    system = "<|system|>\n" + example["system"] + "\n"

     # Create the user prompt and assistant section
    prompt = (
        "<|user|>\n"
        + example["input"]
        + "\n\n<|assistant|>\n"
    )

    # Add a newline to the preferred answer
    chosen = example["chosen"] + "\n"

    # Add a newline to the rejected answer
    rejected = example["rejected"] + "\n"

    # Return the formatted prompt, chosen answer, and rejected answer
    return {
        "prompt": system + prompt,
        "chosen": chosen,
        "rejected": rejected,
    }


# Apply formatting to the dataset and select relatively short answers
# Load the Intel Orca DPO preference dataset
dpo_dataset = load_dataset(
    "argilla/distilabel-intel-orca-dpo-pairs",
    split="train"
)

# Filter the dataset to keep high-quality and useful examples
dpo_dataset = dpo_dataset.filter(
    lambda r:

    # Remove examples where both answers are considered equal
        r["status"] != "tie"

    # Keep examples where the preferred answer has a score of at least 8
        and r["chosen_score"] >= 8

    # Remove examples that are already present in the GSM8K training set
        and not r["in_gsm8k_train"]
)

# Apply the formatting function to every example
dpo_dataset = dpo_dataset.map(

    # Convert each example into prompt, chosen, and rejected format
    format_prompt,

     # Remove the original columns after formatting
    remove_columns=dpo_dataset.column_names
)

# Display the formatted DPO dataset
dpo_dataset

In [ ]:
       # Model Quantization

In [ ]:
# Import AutoPeftModelForCausalLM to load the saved QLoRA model
from peft import AutoPeftModelForCausalLM

# Import quantization configuration and tokenizer tools
from transformers import BitsAndBytesConfig, AutoTokenizer

# 4-bit quantization configuration - Q in QLoRA

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                  # Use 4-bit precision model loading
    bnb_4bit_quant_type="nf4",          # Quantization type
    bnb_4bit_compute_dtype="float16",   # Compute dtype
    bnb_4bit_use_double_quant=True,     # Apply nested quantization
)

# Load the saved QLoRA model
model = AutoPeftModelForCausalLM.from_pretrained(

   # Path to the saved QLoRA model
    "TinyLlama-1.1B-qlora",

    # Reduce CPU memory usage while loading
    low_cpu_mem_usage=True,

   # Automatically use available devices
    device_map="auto",

   # Apply the 4-bit quantization configuration
    quantization_config=bnb_config,
)

# Merge the LoRA weights with the base model
merged_model = model.merge_and_unload()


# Specify the original TinyLlama model name
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# Load the tokenizer for the TinyLlama model
tokenizer = AutoTokenizer.from_pretrained(

    # Use the specified model
    model_name,

    # Allow custom tokenizer code if required
    trust_remote_code=True
)

# Use the EOS token as the padding token
tokenizer.pad_token = tokenizer.eos_token

# Add padding to the left side of the input
tokenizer.padding_side = "left"

In [ ]:
# Import LoRA configuration and functions for parameter-efficient training
from peft import (

    # Defines the LoRA settings
    LoraConfig,

     # Prepares the quantized model for training
    prepare_model_for_kbit_training,

    # Adds LoRA adapters to the model
    get_peft_model
)

# Prepare LoRA configuration

peft_config = LoraConfig(

     # Set the LoRA scaling factor
    lora_alpha=32,

    # Set the dropout rate for LoRA layers
    lora_dropout=0.1,

     # Set the rank of the LoRA matrices
    r=64,

    # Do not train bias parameters
    bias="none",

     # Specify that the task is causal language modeling
    task_type="CAUSAL_LM",

    # Specify the model layers where LoRA adapters will be added
    target_modules=[
        "k_proj",
        "gate_proj",
        "v_proj",
        "up_proj",
        "q_proj",
        "o_proj",
        "down_proj"
    ]
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Add LoRA adapter to the model
model = get_peft_model(model, peft_config)

In [ ]:
  # Training Configuration

In [9]:
from trl import DPOConfig

output_dir = "./results"

training_arguments = DPOConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    max_steps=200,
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True,
    warmup_ratio=0.1
)

In [ ]:
  # Training

In [ ]:
# Import DPOTrainer for Direct Preference Optimization training
from trl import DPOTrainer

# Create the DPO trainer
dpo_trainer = DPOTrainer(

    # Use the model for DPO fine-tuning
    model,

    # Use the DPO training settings
    args=training_arguments,

    # Use the formatted preference dataset
    train_dataset=dpo_dataset,

    # Use the tokenizer to process the text
    tokenizer=tokenizer,

    # LoRA configuration can be enabled if required
    # peft_config=peft_config,

    # Control how strongly preference differences affect training
    beta=0.1,

    # Set the maximum length of the prompt
    max_prompt_length=512,

    # Set the maximum total sequence length
    max_length=512,
)

# Start DPO fine-tuning
dpo_trainer.train()

# Save the trained DPO adapter weights
dpo_trainer.model.save_pretrained(
    "TinyLlama-1.1B-dpo-qlora"
)

In [ ]:
# Import PeftModel to load and merge LoRA adapters
from peft import PeftModel

# Load the saved SFT QLoRA model
model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
)

# Merge the SFT LoRA adapter with the base model
sft_model = model.merge_and_unload()

# Load the DPO LoRA adapter on top of the SFT model
dpo_model = PeftModel.from_pretrained(
    sft_model,
    "TinyLlama-1.1B-dpo-qlora",
    device_map="auto",
)

# Merge the DPO LoRA adapter with the SFT model
dpo_model = dpo_model.merge_and_unload()

In [ ]:
from transformers import pipeline

# Create a text-generation pipeline using the final DPO model
pipe = pipeline(
    "text-generation",
    model=dpo_model,
    tokenizer=tokenizer
)

# Give the model a test prompt
prompt = """<|user|>
What is a Large Language Model?
<|assistant|>
"""

# Generate and display the model's response
result = pipe(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7
)

# Print the generated response
print(result[0]["generated_text"])